In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
# Load the Truth file
expansion_truth = pd.read_csv('HeroExpansion.csv')

# Create a lookup set for the "Perfect Match" check
# We keep the casing as it appears in your HeroExpansion.csv
official_names = set(expansion_truth['Hero Name'].str.strip().tolist())

In [3]:
def normalize_hero(name):
    # --- FAILSAFE ---
    if pd.isna(name) or str(name).lower() in ['nan', '', '-', '0', '0.0', 'none']:
        return ""
    
    n = str(name).strip()

    # --- SECTION 1: THE 100% CONFORMANCE CHECK ---
    if n in official_names:
        return n

    # --- SECTION 2: NON-CONFORMING NAME REPAIR ---
    
    # STAGE A: Specific High-Priority Expansion & Spelling Fixes
    # (Fixing these BEFORE global stripping prevents "Dr. Strange" issues)
    n = n.replace('Peter Parker/Aunt May', 'Peter Parker (Civil War)')
    n = n.replace('Dr. Strange', 'Dr. Strange (Illuminati)')
    n = n.replace('Scott Lang, Cat Burgler', 'Scott Lang, Cat Burglar')
    n = n.replace('Super Skrull', 'Super-Skrull')
    n = re.sub(r'Spider\s*man', 'Spider-Man', n, flags=re.IGNORECASE)
    n = n.replace(' The ', ' the ')
    n = n.replace('Elsa Bloodstone', 'Elsa Bloodstone (Marvel Knights)')
    n = n.replace('Peter Parker, Homecoming', 'Peter Parker (Homecoming)')
    n = n.replace('Bruce Banner', 'Bruce Banner (World War Hulk)')
    n = n.replace('Captain Marvel', 'Captain Marvel (Secret Wars Vol. 1)')
    n = n.replace('Ant-Man', 'Ant-Man (Ant-Man)')
    n = n.replace('Black Panther (Avengers)', 'Black Panther (MCU)')

    if n in official_names: return n
    
    # STAGE B: Specific Hero Mapping & Suffix/Title Fixes
    n = n.replace('Wasp', 'Wasp (Ant-Man)')
    n = n.replace('Drax', 'Drax the Destroyer')
    n = n.replace('Daredevil (Iron Fist)', 'Daredevil (Avengers)')
    n = n.replace('Peter Parker', 'Peter Parker (Civil War)')
    n = n.replace('Steve Rogers, Director of Shield', 'Steve Rogers, Director of SHIELD')

    if n in official_names: return n
    
    n = n.replace('Doctor Strange (Doctor Strange and the Shadows of Nightmare)', 'Doctor Strange (Avengers)')

    # STAGE C: Global Formatting (Ampersands, Dots, and "The")
    # Fix ampersands
    n = n.replace(' & ', ' and ')
    if n in official_names: return n

    # Strip dots (Safe to do now that Dr. Strange is handled)
    n = n.replace('.', '')
    if n in official_names: return n

    # Handle "The" prefix to suffix
    if n.startswith('The '):
        n = n[4:] + ", The"
    if n in official_names: return n

    # STAGE D: Late-Stage Mapping (Catching what remains)
    n = n.replace('Black Panther', 'Black Panther (Illuminati)')
    if n in official_names: return n
    
    n = n.replace('Black Widow', 'Black Widow (Avengers)')
    n = n.replace('Wolverine', 'Wolverine (X-Men)')
    n = n.replace('Black Panther (Avengers)', 'Black Panther (MCU)')

    # --- FINAL RE-CHECK TRUTH ---
    if n in official_names:
        return n
            
    print(f"False Name: {name}")
    return n

In [4]:
def parse_bracketed_data(df, col_idx, prefix):
    current_cols = list(df.columns)
    col_name = current_cols[col_idx]
    dropped = []

    # A. Combine text if the next column is an 'Unnamed' neighbor
    main_text = df[col_name].astype(str).str.strip()
    if col_idx + 1 < len(current_cols) and "Unnamed" in current_cols[col_idx + 1]:
        neighbor_name = current_cols[col_idx + 1]
        neighbor_text = df[neighbor_name].astype(str).str.strip()
        parse_text = main_text.where(main_text.str.contains(r'\(', na=False), 
                                     main_text + " " + neighbor_text)
        dropped.append(neighbor_name)
    else:
        parse_text = main_text

    # B. Extract pieces
    # Remove brackets AND remove the rank suffix
    temp_val = parse_text.str.replace(r'\(.*?\)', '', regex=True)
    df[f"{prefix}_Value"] = temp_val.str.replace(r'\d+(?:st|nd|rd|th)', '', regex=True).str.strip()
    
    df[f"{prefix}_Points"] = parse_text.str.extract(r'\((\d*\.?\d+)\)', expand=False)
    df[f"{prefix}_Rank"] = parse_text.str.extract(r'(\d+(?:st|nd|rd|th))', expand=False)

    dropped.append(col_name)
    return dropped

In [5]:
def dynamically_rename_columns(df, official_names):
    hero_count = 1
    rename_map = {}
    cols_to_drop = []
    
    # 1. Standardize headers
    df.columns = [re.sub(r'\s+', ' ', str(col)).strip() for col in df.columns]
    
    current_cols = list(df.columns)
    for i, col in enumerate(current_cols):
        # Already processed this as a neighbor? Skip it.
        if col in cols_to_drop: continue

        # Clean data for consistent regex matching
        df[col] = df[col].astype(str).str.replace(r'[\n\t\r]', ' ', regex=True).str.strip()
        col_clean = col.lower()
        
        # 2. CALL HELPER FOR TRIPLE-THREAT COLUMNS
        if "escape" in col_clean:
            cols_to_drop.extend(parse_bracketed_data(df, i, "Escapes"))
            continue
            
        if "per turn" in col_clean or "ppt" in col_clean:
            cols_to_drop.extend(parse_bracketed_data(df, i, "PPT"))
            continue
            
        if "turn" in col_clean:
            cols_to_drop.extend(parse_bracketed_data(df, i, "Turns"))
            continue

        # 3. WIN/POINTS LOGIC
        if df[col].str.contains(r'(?:Yes|No)\s*\(', na=False).any():
            df['Win_Value'] = df[col].str.extract(r'(Yes|No)', expand=False)
            df['Win_Points'] = pd.to_numeric(df[col].str.extract(r'\((\d*?\.?\d+)\)', expand=False), errors='coerce')
            cols_to_drop.append(col)
            continue 

        # 4. HERO IDENTIFICATION
        if df[col].isin(official_names).any():
            rename_map[col] = f"Hero {hero_count}"
            hero_count += 1


    # 5. Finalize Drops (Add Comments and Setup here)
    for target in ['Comments', 'Setup']:
        if target in df.columns:
            cols_to_drop.append(target)

    df.rename(columns=rename_map, inplace=True)
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore', inplace=True)
    

    # 6. CONSOLIDATE OTHER CLUSTERS
    clusters = {
        'VP_Diff': ['VP Difference', 'VP Diff.'],
        'Fun_Level': ['Fun Level', 'Fun Level (no slash)', 'Fun Lv.'],
        'Name_Clean': ['Name', 'Name(s)'],
        'VP_Pile_1': ['Victory Pile 1', 'Pile 1', 'VP 1'],
        'VP_Pile_2': ['Victory Pile 2', 'Pile 2', 'VP 2'],
        'Total_VP': ['Total VP', 'VP'],
        'Bystanders': [c for c in df.columns if 'stander' in c.lower() or 'bst.' in c.lower()],
        'Attempt': ['Attempt Submitted', 'Attempt Used', 'Used']
    }

    for master, aliases in clusters.items():
        existing = [c for c in aliases if c in df.columns]
        if not existing: continue
        
        master_col = sorted(existing, key=lambda x: df[x].count(), reverse=True)[0]
        df.rename(columns={master_col: master}, inplace=True)
        
        for extra in [e for e in existing if e != master_col]:
            df[master] = df[master].replace(['nan', 'None', ''], np.nan).fillna(df[extra].replace(['nan', 'None', ''], np.nan))
            df.drop(columns=[extra], inplace=True)

    
            
    return df

In [6]:
def impute_victory_points(df):
    vp_cols = ['VP_Pile_1', 'VP_Pile_2', 'VP_Diff','Total_VP']
    
    # 1. Ensure columns exist. If VP_Diff is missing, create it as NaN
    if 'VP_Diff' not in df.columns:
        df['VP_Diff'] = np.nan
        
    for col in vp_cols:
        if col in df.columns:
            # Force to numeric (NaNs become actual np.nan objects)
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 2. Recalculate VP_Diff BEFORE filling with 0
    if 'VP_Pile_1' in df.columns and 'VP_Pile_2' in df.columns:
        # Now this won't KeyError because we ensured it exists in Step 1
        mask = (df['VP_Diff'].isna()) | (df['VP_Diff'] == 0)
        df.loc[mask, 'VP_Diff'] = (df['VP_Pile_1'] - df['VP_Pile_2']).abs()

    # 3. Final Sweep: Fill remaining NaNs with 0 and force INT
    for col in vp_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)
    if 'Total_VP' in df.columns:
        v1 = pd.to_numeric(df.get('VP_Pile_1', 0), errors='coerce').fillna(0)
        v2 = pd.to_numeric(df.get('VP_Pile_2', 0), errors='coerce').fillna(0)
        df['Total_VP'] = df['Total_VP'].replace(['nan', 'None', ''], np.nan).fillna(v1 + v2)

    
            
    return df

In [7]:
def hero_picks(df, official_names):
    # 1. SETUP: Ensure all Hero columns are clean strings
    hero_cols = [c for c in df.columns if 'Hero' in c]
    for col in hero_cols:
        df[col] = df[col].astype(str).str.strip()

    # 2. DEFAULT: Start by assuming Hero 1 and Hero 2 were the picks
    df['Pick 1'] = df.get('Hero 1', np.nan)
    df['Pick 2'] = df.get('Hero 2', np.nan)

    # 3. ATTEMPT & H3/H4 LOGIC
    # Only try to switch to H3/H4 if the 'Attempt' column exists 
    # AND Hero 3/4 actually contain data.
    if 'Attempt' in df.columns and 'Hero 3' in df.columns and 'Hero 4' in df.columns:
        
        # Check row-by-row if Hero 3 and 4 are valid official names
        h3_valid = df['Hero 3'].isin(official_names)
        h4_valid = df['Hero 4'].isin(official_names)
        
        # Identify "Second Attempt" rows
        is_second = df['Attempt'].astype(str).str.lower().str.contains('second', na=False)
        
        # SWITCH CONDITION: 
        # Must be a Second Attempt AND must have valid Hero names in slots 3 & 4
        switch_mask = is_second & h3_valid & h4_valid
        
        df.loc[switch_mask, 'Pick 1'] = df['Hero 3']
        df.loc[switch_mask, 'Pick 2'] = df['Hero 4']

    # 4. DOUBLE CHECK: Verification Step
    # We ensure 'Pick 1' and 'Pick 2' are actually in the official_names list.
    # If they aren't (e.g., they contain 'nan' or garbage), we mark them as null.
    df.loc[~df['Pick 1'].isin(official_names), 'Pick 1'] = np.nan
    df.loc[~df['Pick 2'].isin(official_names), 'Pick 2'] = np.nan

    # 5. FINAL CLEANUP: Drop any column with the word "Hero"
    # This leaves us with just 'Pick 1' and 'Pick 2'
    cols_to_drop = [c for c in df.columns if 'Hero' in c and c not in ['Pick 1', 'Pick 2']]
    df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

    # Optional: Remove rows where we couldn't find a valid Hero Pick
    df = df.dropna(subset=['Pick 1', 'Pick 2'])

    return df

In [8]:
def clean_legndary_data(file_path):
    # 1. LOAD AND INITIAL CLEANING
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    df.columns = df.columns.str.strip() 

    print(f"1. Start: 'Escaped Villains' in columns? {'Points per Turn' in df.columns}")

    # Early purge of obvious header/empty rows
    if 'Name(s)' in df.columns:
        df = df[~df['Name(s)'].isin(['Rank', 'Points', 'Name', 'Name(s)', 'nan', ''])]
    df = df.dropna(how='all')

    # 2. DYNAMIC COLUMN GAP DROPPER (Villain Deck Logic)
    if 'Cards Left in Villain Deck' in df.columns:
        start_idx = df.columns.get_loc('Cards Left in Villain Deck')
        # We only want to drop the specific 'Cards Left' column and any UNNAMED gaps following it
        gap_cols = [df.columns[start_idx]] # Add the villain deck column itself
        for col in df.columns[start_idx + 1:]:
            if 'Unnamed' in col:
                gap_cols.append(col)
            else:
                break # Stop as soon as we hit a real name like "Escaped Villains"
        df.drop(columns=gap_cols, inplace=True)
        print(f"2. After Gap Dropper: Still there? {'Points per Turn' in df.columns}")
            
    # De-duplicate names to prevent column collisions
    new_cols = []
    for col in df.columns:
        count = new_cols.count(col)
        new_cols.append(f"{col}.{count}" if count > 0 else col)
    df.columns = new_cols

    # 3. GENERATE TOURNEY ID
    raw_filename = os.path.basename(file_path).replace(".csv", "")
    clean_name = re.sub(r'W\d+|Week\s*\d+|Results|Copy\s*of\s*', '', raw_filename, flags=re.IGNORECASE)
    tourney_id = clean_name.replace('-', '').replace('  ', ' ').strip().replace(' ', '')
    
    # 4. DYNAMIC RENAMING & EXTRACTION (The Magnet + Regex)
    # This now handles: Hero Picks, Turns, Turn Rank, Win_Value, and Win_Points
    df = dynamically_rename_columns(df, official_names)
    print(f"3. After Rename Function: Still there? (or 'Escapes')? {any(x in df.columns for x in ['Points per Turn'])}")

    # 5. ELIMINATION POD FIX
    def convert_elim_pods(val):
        val_str = str(val).strip().upper()
        if val_str.startswith('E'):
            try:
                # E-Pods are 101, 102, etc.
                return int(re.search(r'\d+', val_str).group()) + 100
            except:
                return np.nan
        return pd.to_numeric(val_str, errors='coerce')
    
    if 'Pod' in df.columns:
        # Fill the Pod number down into the NaNs created by merged cells
        df['Pod'] = df['Pod'].replace(['-', '0', '0.0', 'nan', 'None'], np.nan)
        df['Pod'] = df['Pod'].ffill()
        df['Pod'] = df['Pod'].apply(convert_elim_pods)

    # 6. CRITICAL: DROP GHOST ROWS & DQ ENTRIES
    # We use 'Win_Value' and 'Win_Points' as the "Anchor of Truth"
    if 'Win_Value' in df.columns:
        # Drop if Win_Value is empty OR if the extraction in #4 failed to find points
        df = df[
            df['Win_Value'].notna() & 
            ~df['Win_Value'].astype(str).str.lower().isin(['nan', 'none', '-', '']) &
            df['Win_Points'].notna()
        ]
        print(f"4. After Row Purge: Row count is {len(df)}")

    # Ensure Pod is a clean integer after the drop
    if 'Pod' in df.columns:
        df['Pod'] = df['Pod'].fillna(0).astype(int)

    # 7. UNIQUE ID GENERATION
    # Uses the cleaned Tourney Name + the filled Pod number
    df['Unique_Pod_ID'] = tourney_id + "_Pod_" + df['Pod'].astype(str)

    # Final cleanup of strings before returning
    df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)

    df = impute_victory_points(df)
    df = hero_picks(df, official_names)
    print(f"5. Final Columns: {df.columns.tolist()}")# ... after VP recalculation ...

    if 'Bystanders' in df.columns:
        df['Bystanders'] = df['Bystanders'].replace(['nan', 'None', ''], np.nan).fillna(0)
    if 'Win?' in df.columns:
        df = df[df['Win?'] != '-'].copy()

    
    return df

In [9]:
import glob

# Grabs every CSV file in the "data" folder
file_list = glob.glob("dirty results/*.csv")

all_weeks = []
for file in file_list:
    print(f"Processing: {file}") # This will pinpoint the broken file
    cleaned_df = clean_legndary_data(file)
    all_weeks.append(cleaned_df)

master_df = pd.concat(all_weeks, ignore_index=True)

# Keep columns only if they have 200 or more non-null values
master_df = master_df.dropna(axis=1, thresh=200)

Processing: dirty results\F22W1.csv
1. Start: 'Escaped Villains' in columns? True
3. After Rename Function: Still there? (or 'Escapes')? False
4. After Row Purge: Row count is 66
5. Final Columns: ['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 16', 'VP_Diff', 'Bystanders', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2']
Processing: dirty results\F22W2.csv
1. Start: 'Escaped Villains' in columns? True
3. After Rename Function: Still there? (or 'Escapes')? False
4. After Row Purge: Row count is 61
5. Final Columns: ['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 16', 'VP_Diff', 'Bystanders', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 

C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To r

5. Final Columns: ['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 16', 'VP_Diff', 'Bystanders', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2']
Processing: dirty results\F22W5.csv
1. Start: 'Escaped Villains' in columns? True
3. After Rename Function: Still there? (or 'Escapes')? False
4. After Row Purge: Row count is 31
5. Final Columns: ['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 16', 'VP_Diff', 'Bystanders', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2']
Processing: dirty results\F22W6.csv
1. Start: 'Escaped Villains' in columns? True
3. Af

C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To r

5. Final Columns: ['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 16', 'VP_Diff', 'Bystanders', 'Attempt', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2']
Processing: dirty results\F23W10.csv
1. Start: 'Escaped Villains' in columns? True
2. After Gap Dropper: Still there? True
3. After Rename Function: Still there? (or 'Escapes')? False
4. After Row Purge: Row count is 16
5. Final Columns: ['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 16', 'VP_Diff', 'Bystanders', 'Attempt', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2']
Processing: dirty result

...

C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To r

5. Final Columns: ['Pod', 'Rank', 'Points Earned', 'Total Points', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 18', 'VP_Diff', 'Bystanders', 'Cards Undercover', 'Attempt', 'Fun_Level', 'Results Of Week:', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2']
Processing: dirty results\Su22W1.csv
1. Start: 'Escaped Villains' in columns? True
3. After Rename Function: Still there? (or 'Escapes')? False
4. After Row Purge: Row count is 52
5. Final Columns: ['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 16', 'VP_Diff', 'Bystanders', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2']
Pr

C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)
C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To r

5. Final Columns: ['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 16', 'VP_Diff', 'Bystanders', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2']
Processing: dirty results\Su22W7.csv
1. Start: 'Escaped Villains' in columns? True
3. After Rename Function: Still there? (or 'Escapes')? False
4. After Row Purge: Row count is 4
5. Final Columns: ['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'Unnamed: 16', 'VP_Diff', 'Bystanders', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2']
Processing: dirty results\Su22W8.csv
1. Start: 'Escaped Villains' in columns? True
3. A

C:\Users\beers\AppData\Local\Temp\ipykernel_33308\1912721169.py:80: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace(['nan', 'None', 'nan.0', '0.0'], np.nan, inplace=True)


In [10]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3410 entries, 0 to 3409
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Pod                3410 non-null   int32  
 1   Rank               3410 non-null   object 
 2   Name_Clean         3410 non-null   object 
 3   VP_Pile_1          3410 non-null   int32  
 4   VP_Pile_2          3410 non-null   int32  
 5   Total_VP           3410 non-null   int32  
 6   Win?               3410 non-null   object 
 7   Total Rank Points  3410 non-null   object 
 8   VP_Diff            3410 non-null   int32  
 9   Bystanders         3410 non-null   object 
 10  Fun_Level          3410 non-null   object 
 11  Win_Value          3410 non-null   object 
 12  Win_Points         3410 non-null   float64
 13  Turns_Value        3410 non-null   object 
 14  Turns_Points       3410 non-null   object 
 15  Turns_Rank         3410 non-null   object 
 16  Escapes_Value      3410 

In [11]:
# Force pandas to show every single column name regardless of length
print(master_df.columns.tolist())


['Pod', 'Rank', 'Name_Clean', 'VP_Pile_1', 'VP_Pile_2', 'Total_VP', 'Win?', 'Total Rank Points', 'VP_Diff', 'Bystanders', 'Fun_Level', 'Win_Value', 'Win_Points', 'Turns_Value', 'Turns_Points', 'Turns_Rank', 'Escapes_Value', 'Escapes_Points', 'Escapes_Rank', 'PPT_Value', 'PPT_Points', 'PPT_Rank', 'Unique_Pod_ID', 'Pick 1', 'Pick 2', 'Attempt', 'Points Earned', 'Total Points', 'Cards Undercover']


In [12]:
# Filter for rows where Victory Pile 1 is null, then get unique Pod IDs
missing_vp_pods = master_df[master_df['Turns_Points'].isnull()]['Unique_Pod_ID'].unique()

# Print the list
print("Pods with missing Victory Pile 1 data:")
for pod_id in missing_vp_pods:
    print(pod_id)

Pods with missing Victory Pile 1 data:


In [13]:
master_df.to_csv('tourney_results_cleaned.csv', index=False)

In [14]:
# Filter for any ID ending in 'Pod_0'
ghost_pods = master_df[master_df['Unique_Pod_ID'].str.endswith('Sp23_Pod_7', na=False)]

# View everything in those rows
# I'm including Setup and Name_Clean to help you identify which files they came from
pd.set_option('display.max_columns', None) # Ensure we see all columns
print(f"Total Ghost Rows found: {len(ghost_pods)}")
print(ghost_pods.head(20))

Total Ghost Rows found: 7
      Pod Rank                     Name_Clean  VP_Pile_1  VP_Pile_2  Total_VP  \
2205    7    1                     JustHarold         32         50        82   
2206    7    3                         Jinjoz         52         32        84   
2207    7    4  Captain Cardone & Paisley5002         67         31        98   
2208    7    5      Silverfox & Blushingether         35         27        62   
2209    7    7                      Frank7666         34         37        71   
2210    7    8                       Bmcaleer         55         41        96   
2211    7    9                 FlipFlopManiac         21         43        64   

     Win? Total Rank Points  VP_Diff Bystanders Fun_Level Win_Value  \
2205  1st              29.5       18          7       3/5       Yes   
2206  1st              26.5       20          6       3/5       Yes   
2207  1st                26       36          6       3/5       Yes   
2208  1st                26        8     

In [15]:
dfError=pd.read_csv('dirty results/Sp24W1.csv')

In [16]:
dfError.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173 entries, 0 to 172
Data columns (total 37 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Pod                         14 non-null     float64
 1   Setup                       12 non-null     object 
 2   Rank                        168 non-null    float64
 3   Points Earned               124 non-null    object 
 4   Total Points                124 non-null    float64
 5   Name(s)                     124 non-null    object 
 6   Victory
Pile 1              127 non-null    object 
 7   Victory
Pile 2              124 non-null    object 
 8   Total
VP                    124 non-null    object 
 9   Win?                        125 non-null    object 
 10  Unnamed: 10                 125 non-null    object 
 11  Turns                       125 non-null    object 
 12  Unnamed: 12                 125 non-null    object 
 13  Escaped Villains            125 non

In [17]:
dfError.head()

,Pod,Setup,Rank,Points Earned,Total Points,Name(s),Victory\nPile 1,Victory\nPile 2,Total\nVP,Win?,Unnamed: 10,Turns,Unnamed: 12,Escaped Villains,Unnamed: 14,Points per Turn,Unnamed: 16,Total Rank Points,Unnamed: 18,VP Difference,Bystanders Rescued,Cards Left in Villain Deck,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Attempt 1 Hero Choices,Unnamed: 29,Attempt 2 Hero Choices,Unnamed: 31,Attempt Submitted,Fun Level,Comments,Unnamed: 35,Results Of Week:
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Strikes,Twists,Villains,Henchmen,Bystanders,Other,Total,1,2,1,2,NaN,NaN,NaN,NaN,1.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Rank,Points,Rank,Points,Rank,Points,Rank,Points,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.0,Mysterio's HYDRA Hypnosis,1.0,(+11),11.0,JB007,101,23,124,1st,Yes (6.5),1st,24 (11),7th,2 (3.5),1st,5.167 (11),32,NaN,78,27,2,1,5,5,1,0,14,Mockingbird,Wolverine (X-Men),I did not play a second attempt.,I did not play a second attempt.,First attempt,4,This was a blast! Lopsided with player 1 whoop...,NaN,NaN
3,NaN,NaN,2.0,(+10),10.0,WingedCrusader,66,32,98,1st,Yes (6.5),2nd,26 (10),3rd,1 (7.5),5th,3.769 (7),31,NaN,34,3,0,4,6,3,1,0,14,"Miek, the Unhived",Mockingbird,"Miek, the Unhived",Mockingbird,Second attempt,5,Okay the first time around I was just getting ...,NaN,NaN
4,NaN,NaN,3.0,(+9),9.0,Swagolas,105,49,154,1st,Yes (6.5),6th,33 (6),3rd,1 (7.5),3rd,4.667 (9),29,NaN,56,18,0,2,3,1,1,0,7,Mockingbird,Deathlok,Mockingbird,Deathlok,First attempt,5,That was awesome! I loved the undercover keywo...,NaN,NaN


## 